In [30]:
import pandas as pd
import numpy as np

# Ορίζουμε seed για να παίρνεις πάντα τα ίδια ακριβώς "τυχαία" δεδομένα
np.random.seed(42)

# 1. Δημιουργία dim_employee (100 υπάλληλοι συνολικά: 12 managers + 88 απλοί υπάλληλοι)
num_employees = 100
empl_ids = [f"EMP_{i:03d}" for i in range(1, num_employees + 1)]

# Οι πρώτοι 12 είναι managers, οι υπόλοιποι 88 είναι υφιστάμενοι
manager_ids = empl_ids[:12]
staff_ids = empl_ids[12:]

# Μισθοί managers (κυμαίνονται μεταξύ 3.200€ και 6.500€ - μερικοί επίτηδες χαμηλοί!)
manager_salaries = np.random.randint(3200, 6500, size=12)

# Μισθοί υφισταμένων (κυμαίνονται μεταξύ 1.500€ και 3.500€)
staff_salaries = np.random.randint(1500, 3500, size=88)

all_salaries = np.concatenate([manager_salaries, staff_salaries])

# Προσθέτουμε και μερικές έξτρα στήλες (όπως στα πραγματικά δεδομένα)
# για να φανεί η αξία του dim_employee[['empl_id', 'salary']] στον κώδικά σου!
departments = np.random.choice(['Data', 'Engineering', 'Product', 'Marketing', 'Sales'], size=num_employees)
hire_dates = pd.date_range(start='2020-01-01', periods=num_employees, freq='15D')

dim_employee = pd.DataFrame({
    'empl_id': empl_ids,
    'salary': all_salaries,
    'department': departments,
    'hire_date': hire_dates
})

# 2. Δημιουργία map_employee_hierarchy (Ποιος αναφέρεται σε ποιον)
# Κάθε ένας από τους 88 υπαλλήλους ανατίθεται τυχαία σε έναν από τους 12 managers
assigned_managers = np.random.choice(manager_ids, size=len(staff_ids))

map_employee_hierarchy = pd.DataFrame({
    'empl_id': staff_ids,
    'manager_empl_id': assigned_managers
})


In [31]:
dim_employee.head()

,empl_id,salary,department,hire_date
0,EMP_001,6374,Engineering,2020-01-01
1,EMP_002,4060,Engineering,2020-01-16
2,EMP_003,4494,Marketing,2020-01-31
3,EMP_004,4330,Marketing,2020-02-15
4,EMP_005,4295,Data,2020-03-01


In [32]:
map_employee_hierarchy.head()

,empl_id,manager_empl_id
0,EMP_013,EMP_012
1,EMP_014,EMP_008
2,EMP_015,EMP_011
3,EMP_016,EMP_006
4,EMP_017,EMP_008


### solution

In [33]:
employee= dim_employee[['empl_id', 'salary']]
merged = pd.merge(map_employee_hierarchy, employee, how='inner', on='empl_id')

merged = pd.merge(merged, employee, how='inner', left_on='manager_empl_id', 
                  right_on='empl_id', suffixes=['_emp', '_mgr'])

merged = merged.drop(columns='manager_empl_id')
merged= merged.rename(columns={'empl_id_emp':'empl_id',
                               'salary_emp':'empl_sal',
                               'salary_mgr':'mgr_sal',
                               'empl_id_mgr':'mgr_id'})
merged.head()
df = merged.groupby('mgr_id')['empl_sal'].mean().round().astype(int).reset_index(name='avg_emp_sal')
df= pd.merge(merged,df, how='inner', on='mgr_id')

final_query = (
    df.query("mgr_sal< avg_emp_sal*2")
    [['mgr_id','mgr_sal']]
    .drop_duplicates()   
)
final_query

,mgr_id,mgr_sal
2,EMP_011,3530
5,EMP_009,3666
6,EMP_004,4330
9,EMP_010,4438
12,EMP_007,4838
13,EMP_002,4060
14,EMP_003,4494
16,EMP_005,4295


### solution

In [34]:
# 1. Select only required columns to optimize memory
dim_employee_clean = dim_employee[['empl_id', 'salary']]

# 2. Map each employee to their respective manager ID
combined = pd.merge(dim_employee_clean, map_employee_hierarchy, how='inner', on='empl_id')

# 3. Fetch manager salaries via a second merge (self-join pattern)
combined = pd.merge(
    combined, 
    dim_employee_clean, 
    how='inner', 
    left_on='manager_empl_id', 
    right_on='empl_id', 
    suffixes=['_emp', '_mgr']
)

# 4. Drop redundant key column
combined = combined.drop(columns=['manager_empl_id'])

# 5. Calculate the average subordinate salary per manager
avg = combined.groupby('empl_id_mgr')['salary_emp'].mean().reset_index(name='mean_emp_sal')

# 6. Attach team average back to the main dataset
combined = pd.merge(combined, avg, how='inner', on='empl_id_mgr')

# 7. Filter managers earning < 2x their team's average and deduplicate
result = (
    combined.loc[
        combined['salary_mgr'] < (combined['mean_emp_sal'] * 2),
        ['empl_id_mgr', 'salary_mgr', 'mean_emp_sal']
    ]
    .drop_duplicates()
    .round(2)
)

result


,empl_id_mgr,salary_mgr,mean_emp_sal
2,EMP_011,3530,2397.88
5,EMP_009,3666,2476.50
6,EMP_004,4330,2716.11
9,EMP_010,4438,2477.20
12,EMP_007,4838,2623.00
13,EMP_002,4060,2286.14
14,EMP_003,4494,2508.43
16,EMP_005,4295,2608.50
